In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.__version__

'3.0.0'

In [3]:
df=pd.read_csv(r"C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\movies_courses_watch_history.csv")

In [4]:
df.head()

,user_id,time,month_diff,movie_id,title,domain,metadata,combined
0,MHxPC130442623,9.0,13,CB22x,HarvardX CB22x,Biochemistry,Fundamentals of biochemistry including protein...,Fundamentals of biochemistry including protein...
1,MHxPC130442623,9.0,15,CS50x,HarvardX CS50x,Computer Science,Introduction to computer science covering algo...,Introduction to computer science covering algo...
2,MHxPC130275857,16.0,11,CB22x,HarvardX CB22x,Biochemistry,Fundamentals of biochemistry including protein...,Fundamentals of biochemistry including protein...
3,MHxPC130275857,16.0,16,CS50x,HarvardX CS50x,Computer Science,Introduction to computer science covering algo...,Introduction to computer science covering algo...
4,MHxPC130275857,16.0,13,ER22x,HarvardX ER22x,Engineering,Engineering fundamentals with interdisciplinar...,Engineering fundamentals with interdisciplinar...


In [5]:
df.shape

(741137, 8)

In [6]:
df1=pd.read_csv(r"C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\movies_courses.csv")

In [7]:
df1.sample(5)

,title,domain,metadata,id,combined
10933,Tell Me Who I Am,Movie,"Documentaries In this documentary, Alex trusts...",s6018,"Documentaries In this documentary, Alex trusts..."
11240,The Edge of Seventeen,Movie,"Comedies, Dramas When Nadine's best (and only)...",s6325,"Comedies, Dramas When Nadine's best (and only)..."
2071,leadership principles for software engineers,Software Development,"People Management,Business Strategy,Cloud Engi...",n2232,"People Management,Business Strategy,Cloud Engi..."
1956,java class library,Software Development,"Recursively Enumerable Set,Iterator,Linearity,...",n2099,"Recursively Enumerable Set,Iterator,Linearity,..."
3385,microsoft future ready: designing and implemen...,NaN,.,n3898,. microsoft future ready: designing and impl...


In [8]:
df1.rename(columns={'id':'movie_id'},inplace=True)

In [9]:
df1['movie_id'].value_counts()

movie_id
n0            1
n1            1
n2            1
n3            1
n4            1
             ..
movie_0996    1
movie_0997    1
movie_0998    1
movie_0999    1
movie_1000    1
Name: count, Length: 13703, dtype: int64

In [10]:
df['user_id'].value_counts()

user_id
user_06589        23
user_05244        22
user_02666        22
user_06554        22
user_05600        22
                  ..
MHxPC130591057     1
MHxPC130226305     1
MHxPC130030805     1
MHxPC130184108     1
MHxPC130359782     1
Name: count, Length: 486532, dtype: int64

In [11]:
# All are movies only

In [ ]:
# As i need to give importance for each movie based on recent history and watch time .
# By this we need to generate the embedding for each user's row,then we need to group the embeding per user and do weighted average with respect to time
# of them to get final embediings per user


## Steps
# i)create embeddings per user history
# ii) group the history based on each user
# iii) do the weighted sum based on time and month_diff

In [13]:
df.shape

(741137, 8)

In [14]:
df.isnull().sum()

user_id            0
time          174513
month_diff         0
movie_id           0
title              0
domain             0
metadata           0
combined           0
dtype: int64

In [15]:
df1.isnull().sum()

title          0
domain      2338
metadata       0
movie_id       0
combined       0
dtype: int64

In [16]:
df.duplicated().sum()

np.int64(381)

In [ ]:
# As the data is very high we will drop this 400 rows.

In [17]:
df.shape

(741137, 8)

In [18]:
df.drop_duplicates(inplace=True)

In [19]:
df.shape

(740756, 8)

In [20]:
df['domain'].value_counts()

domain
Computer Science          169621
Programming               124440
Physics                   112242
Electrical Engineering     62674
Engineering                57406
Movie                      44451
Biochemistry               30002
Economics                  27870
TV Series                  25308
Biology                    21009
Chemistry                  20351
Documentary                13642
Stand-up Comedy            11469
Renewable Energy            9477
Mechanical Engineering      5665
Limited Series              5129
Name: count, dtype: int64

In [21]:
m = df1['combined'].fillna('').str.split(" ")

In [ ]:
p=0
for x in range(len(m)):
        if len(m[x])>100:
           p+=1
print(p)
# As this is all the string with more than length 100 ,there are 700 strings with long strings. 

692


In [23]:
df.shape

(740756, 8)

In [24]:
df1.shape

(13703, 5)

In [25]:
## User History
##    ↓
## User Embedding
##    ↓
## Query Vector
##    ↓
## FAISS (chunk retrieval)
##    ↓
## Top chunks
##    ↓
## Aggregate → movie scores
##    ↓
## Remove watched
##    ↓
## Hybrid scoring (BM25 + semantic)
##    ↓
## Cross-encoder re-ranking
##    ↓
## Top-K recommendations
##    ↓
## Evaluation (Precision@K, Recall@K, MRR)

# Chunking-->As it required because long senetence may loose some of the semantic meaning. 

In [26]:
import re
# semantic chunking 
def semantic_split(text):
    sentences = re.split(r'(?<=[.!?]) +', text)
    return sentences
# semantic + sliding window chunking
def semantic_sliding_chunks(text, max_words=100, overlap=30):
    sentences = semantic_split(text)
    
    chunks = []
    current_chunk = []
    current_len = 0
    
    for sent in sentences:
        words = sent.split()
        
        if current_len + len(words) <= max_words:
            current_chunk.append(sent)
            current_len += len(words)
        else:
            
            chunks.append(" ".join(current_chunk))
           
            overlap_words = " ".join(current_chunk).split()[-overlap:]
            current_chunk = [" ".join(overlap_words), sent]
            current_len = len(overlap_words) + len(words)
    
    
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks

In [27]:

df['time']=df['time'].fillna(df['time'].mean())

In [28]:
df.head(1)

,user_id,time,month_diff,movie_id,title,domain,metadata,combined
0,MHxPC130442623,9.0,13,CB22x,HarvardX CB22x,Biochemistry,Fundamentals of biochemistry including protein...,Fundamentals of biochemistry including protein...


In [29]:
df1.head(1)

,title,domain,metadata,movie_id,combined
0,machine learning specialization,Machine Learning,"Decision Trees, Artificial Neural Network, Log...",n0,"Decision Trees, Artificial Neural Network, Log..."


In [30]:
# do sepaarte training for both of them:
all_chunks = []
chunk_metadata = []

for idx, row in df1.iterrows():
    chunks = semantic_sliding_chunks(row['combined'])
    
    for chunk in chunks:
        all_chunks.append(chunk)
        
        chunk_metadata.append({
            "row_idx": idx,
            "id": row['movie_id']})
            

In [31]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
arr = model.encode(
    all_chunks,
    convert_to_tensor=True,
    normalize_embeddings=True
)
item_emb = arr.cpu().numpy()

c:\nihal\python\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2119.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
df['combined'].isnull().sum()

np.int64(0)

In [33]:
user_corpus=df['combined'].to_list()
df['embedding'] = model.encode(
    user_corpus,
    convert_to_tensor=True,
    normalize_embeddings=True
).cpu().numpy().tolist()

In [ ]:
# FAISS retreival

In [34]:
import faiss 
dim=item_emb.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(item_emb)

In [ ]:
import numpy as np
from sklearn.preprocessing import normalize

# weights

df["weight2"] =df["time"] * np.exp(-0.3 * df["month_diff"])


df["weight2"] = df["weight2"].fillna(0)

user_embeddings = {}

for user_id, group in df.groupby("user_id"):
    

    user_emb = np.vstack(group["embedding"].values)

    weights2 = group["weight2"].values.reshape(-1, 1)

    

    # -------- weight --------
    norm2 = np.sum(weights2)
    if norm2 == 0:
        emb2 = np.zeros(user_emb.shape[1])
    else:
        emb2 = np.sum(user_emb * weights2, axis=0) / norm2

    # Normalize (VERY IMPORTANT)
    emb2 = normalize([emb2])[0]
    
    user_embeddings[user_id] = emb2

In [36]:
import pickle
import os

folder_path = r"C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset"
os.makedirs(folder_path, exist_ok=True)

file_path = os.path.join(folder_path, "phase_2_user_emb.pkl")

with open(file_path, "wb") as f:
    pickle.dump(user_embeddings, f)

print("File saved at:", file_path)

File saved at: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\phase_2_user_emb.pkl


In [ ]:
# As Break total embeddings into 5000 chunks so that the ram may not crash during training the model.
folder_path = r"C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\user"
os.makedirs(folder_path, exist_ok=True)

# Number of chunks
num_chunks =5000

chunks = np.array_split(df, num_chunks)

for i, chunk in enumerate(chunks):
    file_path = os.path.join(folder_path, f"phase_2_user_part_{i+1}.pkl")    
    with open(file_path, "wb") as f:
      pickle.dump(chunk, f)
    print(f"Saved: {file_path}")

Saved: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\user\phase_2_user_part_1.pkl
Saved: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\user\phase_2_user_part_2.pkl
Saved: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\user\phase_2_user_part_3.pkl
Saved: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\user\phase_2_user_part_4.pkl
Saved: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\user\phase_2_user_part_5.pkl
Saved: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\user\phase_2_user_part_6.pkl
Saved: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\user\phase_2_user_part_7.pkl
Saved: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\user\phase_2_user_part_8.pkl
Saved: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\user\phase_2_user_part_9.pkl
Saved: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\user\phase_2_user_part_10.pkl
Saved: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\user\phase_2_user_part_11.p

In [ ]:
folder_path = r"C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset"
os.makedirs(folder_path, exist_ok=True)

file_path = os.path.join(folder_path, "phase_2_item.pkl")

df1.to_pickle(file_path)

print("File saved at:", file_path)

File saved at: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\phase_2_item.pkl


In [ ]:
folder_path = r"C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset"
os.makedirs(folder_path, exist_ok=True)

file_path = os.path.join(folder_path, "phase_2_chunk_metadata.pkl")

with open(file_path, "wb") as f:
    pickle.dump(chunk_metadata, f)

print("File saved at:", file_path)

File saved at: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\phase_2_chunk_metadata.pkl


In [ ]:
folder_path = r"C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset"
os.makedirs(folder_path, exist_ok=True)

file_path = os.path.join(folder_path, "phase_2_item_emb.pkl")

with open(file_path, "wb") as f:
    pickle.dump(item_emb, f)

print("File saved at:", file_path)

File saved at: C:\nihal\python\NLP\Reccomender_system\Cleaned_dataset\phase_2_item_emb.pkl


In [42]:
df['user_id'].unique()

<ArrowStringArray>
['MHxPC130442623', 'MHxPC130275857', 'MHxPC130539455', 'MHxPC130088379',
 'MHxPC130198098', 'MHxPC130024894', 'MHxPC130080986', 'MHxPC130063375',
 'MHxPC130094371', 'MHxPC130229084',
 ...
     'user_07781',     'user_08189',     'user_08162',     'user_06109',
     'user_09933',     'user_07500',     'user_02894',     'user_05844',
     'user_09723',     'user_09732']
Length: 486532, dtype: str

In [43]:

from sentence_transformers.util import cos_sim
import torch
user_id = "user_02894"
user_vec = user_embeddings[user_id].reshape(1, -1)
user_vec = torch.tensor(user_vec, dtype=torch.float32)
item_emb = torch.tensor(item_emb, dtype=torch.float32)

In [44]:
D, I = index.search(user_vec, k=20)

In [45]:
# Now we have all embedinngs for users and also items,
# so do retreival and ranking mechanism

In [46]:
!pip install rank_bm25

In [ ]:
# the pipeline which has all steps included in phase-2,which automates in one flow. 

In [47]:
from rank_bm25 import BM25Okapi

def build_bm25(df_items):
    corpus = df_items['combined'].tolist()
    tokenized = [doc.split() for doc in corpus]
    return BM25Okapi(tokenized)

In [48]:
def retrieve_chunks(user_vec, index, k=20):
    user_vec = user_vec.reshape(1, -1)
    D, I = index.search(user_vec, k)
    return D[0], I[0]

In [49]:
from collections import defaultdict

def aggregate_scores(D, I, chunk_data):
    movie_scores = defaultdict(float)

    for idx, score in zip(I, D):
        movie_id = chunk_data[idx]['id']
        movie_scores[movie_id] = max(movie_scores[movie_id], score)

    return movie_scores

In [50]:
def remove_watched(movie_scores, df, user_id):
    watched = set(df[df['user_id']==user_id]['movie_id'])
    return {k:v for k,v in movie_scores.items() if k not in watched}

In [51]:
def hybrid_score(movie_scores, bm25,new_df, query, alpha=0.7):
    bm_scores = bm25.get_scores(query.split())
    final_scores = {}
    for i, row in new_df.iterrows():
        movie_id = row['movie_id']
        sem = movie_scores.get(movie_id, 0)
        bm = bm_scores[i]
        final_scores[movie_id] = alpha * sem + (1 - alpha) * bm
    return final_scores

In [52]:
def build_query(user_id, df, top_n=5):
    user_data = df[df['user_id'] == user_id]
    texts = user_data.sort_values('weight2', ascending=False)['combined'].head(top_n)
    return " ".join(texts)

In [ ]:
# reranking for cross encoder
def rerank(query, movie_ids, df_items, cross_model, batch_size=32):
    docs = df_items.set_index('movie_id').loc[movie_ids]['combined'].tolist()
    pairs = [[query, doc] for doc in docs]
    scores = cross_model.predict(pairs, batch_size=batch_size)
    ranked = sorted(zip(movie_ids, scores), key=lambda x: x[1], reverse=True)
    return [x[0] for x in ranked]

In [ ]:
from sentence_transformers import CrossEncoder
cross_model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2'
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1381.70it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def recommend(user_id, df, df_items, user_vectors, index, chunk_data, bm25, cross_model, k=5):
    user_vec = user_vectors[user_id]

    query=build_query(user_id,df)
    print(query)
    # retrieval
    D, I = retrieve_chunks(user_vec, index)

    # aggregate
    movie_scores = aggregate_scores(D, I, chunk_data)

    # remove watched
    movie_scores = remove_watched(movie_scores, df, user_id)

    # hybrid
    final_scores = hybrid_score(movie_scores, bm25, df_items,query)

    # top candidates
    top_movies = sorted(final_scores, key=final_scores.get, reverse=True)[:10]

    # rerank by cross encoder
    final = rerank("user preference", top_movies, df_items, cross_model)

    return final[:k]

In [56]:
bm=build_bm25(df1)
pred=recommend(user_id,df,df1,user_embeddings,index,chunk_metadata,bm,cross_model)

Adventure TV Series storm kingdom Comedy Thriller TV Series day legend


## Metrics and Evaluation 

In [57]:
def precision_at_k(relevant, predicted, k):
    predicted = predicted[:k]
    return len(set(predicted) & set(relevant)) / k

In [58]:
def recall_at_k(relevant, predicted, k):
    predicted = predicted[:k]
    return len(set(predicted) & set(relevant)) / len(relevant)

In [59]:
def reciprocal_rank(relevant, predicted):
    for i, p in enumerate(predicted):
        if p in relevant:
            return 1 / (i + 1)
    return 0

In [60]:
def get_relevant(user_id, df, k=5):
    user_data = df[df['user_id'] == user_id]
    top_k = user_data.sort_values(by='weight2', ascending=False).head(k)
    return top_k['movie_id'].tolist()

In [61]:
def evaluate(user_id, relevant_items,predicted_items):
  
    p = precision_at_k(relevant_items, predicted_items, k=5)
    r = recall_at_k(relevant_items, predicted_items, k=5)
    mrr = reciprocal_rank(relevant_items, predicted_items)

    return {
        "Precision@5": p,
        "Recall@5": r,
        "MRR": mrr
    }

In [62]:
relevant_items=get_relevant(user_id,df)

In [63]:
pred

['movie_0394', 'movie_0441', 'movie_0517', 'movie_0169', 'movie_0814']

In [ ]:
df = pd.DataFrame()  # start empty

for i in pred:
    new_row = df1[df1['movie_id'] == i]
    df = pd.concat([df, new_row], ignore_index=True)
print(df)

              title     domain   metadata    movie_id  \
13096  dark kingdom  TV Series  Adventure  movie_0394   

                               combined  
13096  Adventure TV Series dark kingdom  
            title     domain         metadata    movie_id  \
13143  day legend  TV Series  Comedy Thriller  movie_0441   

                                   combined  
13143  Comedy Thriller TV Series day legend  
              title     domain metadata    movie_id  \
13219  storm legend  TV Series      War  movie_0517   

                         combined  
13219  War TV Series storm legend  
           title     domain metadata    movie_id                     combined
12871  storm day  TV Series  Western  movie_0169  Western TV Series storm day
                  title     domain   metadata    movie_id  \
13516  adventure legend  TV Series  Adventure  movie_0814   

                                   combined  
13516  Adventure TV Series adventure legend  


In [65]:
df[df['user_id']==user_id]

,user_id,time,month_diff,movie_id,title,domain,metadata,combined,embedding,weight1,weight2
712090,user_02894,125.300000,6,movie_0425,storm kingdom,TV Series,Adventure,Adventure TV Series storm kingdom,"[-0.025716934353113174, 0.04490227997303009, -...",20.883333,20.711951
730772,user_02894,15.059873,1,movie_0441,day legend,TV Series,Comedy Thriller,Comedy Thriller TV Series day legend,"[-0.05633559450507164, 0.034831441938877106, -...",15.059873,11.156628


In [66]:
print(pred)

['movie_0394', 'movie_0441', 'movie_0517', 'movie_0169', 'movie_0814']


In [67]:
fin=evaluate(user_id,relevant_items,pred)
print(fin)

{'Precision@5': 0.2, 'Recall@5': 0.5, 'MRR': 0.5}


In [68]:
df.head()

,user_id,time,month_diff,movie_id,title,domain,metadata,combined,embedding,weight1,weight2
0,MHxPC130442623,9.0,13,CB22x,HarvardX CB22x,Biochemistry,Fundamentals of biochemistry including protein...,Fundamentals of biochemistry including protein...,"[0.003973859362304211, -0.02617398090660572, -...",0.692308,0.182177
1,MHxPC130442623,9.0,15,CS50x,HarvardX CS50x,Computer Science,Introduction to computer science covering algo...,Introduction to computer science covering algo...,"[-0.031119130551815033, -0.02509980835020542, ...",0.600000,0.099981
2,MHxPC130275857,16.0,11,CB22x,HarvardX CB22x,Biochemistry,Fundamentals of biochemistry including protein...,Fundamentals of biochemistry including protein...,"[0.003973859362304211, -0.02617398090660572, -...",1.454545,0.590131
3,MHxPC130275857,16.0,16,CS50x,HarvardX CS50x,Computer Science,Introduction to computer science covering algo...,Introduction to computer science covering algo...,"[-0.031119130551815033, -0.02509980835020542, ...",1.000000,0.131676
4,MHxPC130275857,16.0,13,ER22x,HarvardX ER22x,Engineering,Engineering fundamentals with interdisciplinar...,Engineering fundamentals with interdisciplinar...,"[-0.0042107379995286465, -0.019862225279211998...",1.230769,0.323871


In [69]:


scores = cos_sim(user_vec, item_emb)[0]
top_k = 20
top_results = scores.topk(k=top_k)
from collections import defaultdict

item_scores = defaultdict(float)

for score, idx in zip(top_results.values, top_results.indices):
    meta = chunk_metadata[idx.item()]
    item_id = meta["id"]
    
    item_scores[item_id] = max(item_scores[item_id], score.item())
ranked_items = sorted(item_scores.items(), key=lambda x: x[1], reverse=True)
results = []

for item_id, score in ranked_items[:10]:
    row = df[df['movie_id'] == item_id].iloc[0]
    
    row_data = {
        'title': row['title'],
        'domain': row['domain'],
        'metadata': row['metadata'],
        'score': score
    }
    
    results.append(row_data)

fin = pd.DataFrame(results)

In [70]:
print(fin)

              title     domain               metadata     score
0     storm kingdom  TV Series              Adventure  0.934104
1      legend storm  TV Series                Fantasy  0.810104
2      storm legend  TV Series                    War  0.761334
3      dark kingdom  TV Series              Adventure  0.761242
4       storm night  TV Series  Adventure Documentary  0.758963
5  adventure legend  TV Series        Drama Adventure  0.757547
6        day legend  TV Series        Comedy Thriller  0.748829
7         the storm  TV Series          History Drama  0.740922
8  adventure legend  TV Series              Adventure  0.740219
9         was storm  TV Series                Mystery  0.735289


In [71]:
# As from advanced model

## AS Advanced fixes of this phase-2

In [ ]:
#Problem:Filter Bubble:
# User watches:
#Action movie → Action movie → Action movie
#
#Embedding becomes:
#"pure action vector"
#
#System keeps recommending:
#ONLY action



## solution:Diversity Re-ranking
#
#After you get top-K results, re-rank them
#
#Idea:
#Penalize items that are too similar to each other,by this we can eliminate the items which are too similar.
#Intuition:
#First item → highest relevance
#Next items → must be:
#i)relevant 
#ii)different from already selected items
#
#Formula:
#    
#    Final Score = relevance_score - λ * similarity_with_selected_items
# we are making it slow penalty ,so that it may not delete for ever,rather we are penalizing it.



In [73]:
def make(candidates, K, item_emb, user_vec):
    selected = []
    lambda_ = 0.7

    relevance_scores = cos_sim(user_vec, item_emb)[0]

    candidate_indices = candidates['index'].tolist()

    while len(selected) < K and len(candidate_indices) > 0:
        best_item = None
        best_score = -1e9

        for idx in candidate_indices:
            relevance = relevance_scores[idx].item()

            diversity_penalty = 0
            for sel_idx in selected:
                sim = cos_sim(
                    item_emb[idx].unsqueeze(0),
                    item_emb[sel_idx].unsqueeze(0)
                )[0][0].item()
                
                diversity_penalty = max(diversity_penalty, sim)

            score = lambda_ * relevance - (1 - lambda_) * diversity_penalty

            if score > best_score:
                best_score = score
                best_item = idx

        selected.append(best_item)
        candidate_indices.remove(best_item)

    return selected

In [74]:
top_k = 20
top_res = scores.topk(k=top_k)
results = []

for score, idx in zip(top_res.values, top_res.indices):
    results.append({
        'index': idx.item(),
        'score': score.item()
    })

candidates = pd.DataFrame(results)
selected_indices = make(candidates, 100, item_emb, user_vec)
results = []

for idx in selected_indices:
    row = df.iloc[idx]

    results.append({
        'title': row['title'],
        'domain': row['domain'],
        'metadata': row['metadata'],
        'score': candidates[candidates['index'] == idx]['score'].values[0]
    })

fin = pd.DataFrame(results)
fin=fin.sort_values(by="score",ascending=False)

In [ ]:
fin=pd.DataFrame(fin)
print(fin)

              title            domain  \
0    HarvardX CS50x  Computer Science   
2    HarvardX CS50x  Computer Science   
11   HarvardX ER22x       Engineering   
6   HarvardX PH278x           Physics   
4    HarvardX CB22x      Biochemistry   
3    HarvardX CS50x  Computer Science   
1    HarvardX CS50x  Computer Science   
14  HarvardX PH207x           Physics   
19   HarvardX ER22x       Engineering   
9   HarvardX PH207x           Physics   
8    HarvardX CS50x  Computer Science   
13   HarvardX CS50x  Computer Science   
7   HarvardX PH207x           Physics   
17   HarvardX CS50x  Computer Science   
15   HarvardX CS50x  Computer Science   
10  HarvardX PH207x           Physics   
12   HarvardX CS50x  Computer Science   
18   HarvardX ER22x       Engineering   
16   HarvardX ER22x       Engineering   
5    HarvardX CS50x  Computer Science   

                                             metadata     score  
0   Introduction to computer science covering algo...  0.934104  
2   In

In [76]:
fin['domain'].value_counts()

domain
Computer Science    10
Physics              5
Engineering          4
Biochemistry         1
Name: count, dtype: int64